**Retrieval Augmented Generation (RAG)**<br>
Retrieval Augmented Generation (RAG) is a method for generating text using additional information fetched from an external data source, which can greatly increase the accuracy of the response. 

In [1]:
import os

from getpass import getpass
import nest_asyncio

from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv()
CO_API_KEY = os.environ['CO_API_KEY'] or getpass("Enter your Cohere API key: ")

In [27]:
import cohere
co = cohere.Client(api_key=CO_API_KEY)
co.chat(
    model="command-a-03-2025",
    message="Where do the tallest penguins live?",
    documents=[
        {
            "title": "Tall penguins",
            "snippet": "Emperor penguins are the tallest.",
        },
        {
            "title": "Penguin habitats",
            "snippet": "Emperor penguins only live in Antarctica.",
        },
        {
            "title": "What are animals?",
            "snippet": "Animals are different from plants.",
        },
    ],
)

NonStreamedChatResponse(text='The tallest penguins, Emperor penguins, live in Antarctica.', generation_id='08e2aded-1bb9-49be-b9e7-b817db4edf88', response_id='c289981f-9c2b-4e20-b415-2d467cf2796a', citations=[ChatCitation(start=22, end=38, text='Emperor penguins', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=48, end=59, text='Antarctica.', document_ids=['doc_1'], type='TEXT_CONTENT')], documents=[{'id': 'doc_0', 'snippet': 'Emperor penguins are the tallest.', 'title': 'Tall penguins'}, {'id': 'doc_1', 'snippet': 'Emperor penguins only live in Antarctica.', 'title': 'Penguin habitats'}], is_search_required=None, search_queries=None, search_results=None, finish_reason='COMPLETE', tool_calls=None, chat_history=[UserMessage(role='USER', message='Where do the tallest penguins live?', tool_calls=None), ChatbotMessage(role='CHATBOT', message='The tallest penguins, Emperor penguins, live in Antarctica.', tool_calls=None)], meta=ApiMeta(api_version=ApiMetaApiVersion(version=

In [5]:
{
  "text": "The tallest penguins, Emperor penguins, live in Antarctica.",  
  "citations": [  
    {"start": 22, "end": 38, "text": "Emperor penguins", "document_ids": ["doc_0"]},  
    {"start": 48, "end": 59, "text": "Antarctica.", "document_ids": ["doc_1"]}  
  ],  
  "documents": [  
    {"id": "doc_0", "title": "Tall penguins", "snippet": "Emperor penguins are the tallest.", "url": ""},  
    {"id": "doc_1", "title": "Penguin habitats", "snippet": "Emperor penguins only live in Antarctica.", "url": ""}  
  ] 
}

{'text': 'The tallest penguins, Emperor penguins, live in Antarctica.',
 'citations': [{'start': 22,
   'end': 38,
   'text': 'Emperor penguins',
   'document_ids': ['doc_0']},
  {'start': 48, 'end': 59, 'text': 'Antarctica.', 'document_ids': ['doc_1']}],
 'documents': [{'id': 'doc_0',
   'title': 'Tall penguins',
   'snippet': 'Emperor penguins are the tallest.',
   'url': ''},
  {'id': 'doc_1',
   'title': 'Penguin habitats',
   'snippet': 'Emperor penguins only live in Antarctica.',
   'url': ''}]}

**Using RAG to Internet Search**<br>
we will use the three step RAG workflow to finally settle the score between the notorious boy bands Backstreet Boys and NSYNC.

**Step 1: Generating search queries**<br>
Calling the Chat API with the search_queries_only parameter set to True will return a list of search queries. In the example below, we ask the model to suggest some search queries that would be useful when answering the question.<br>
**Step 2: Using a tool**<br>
Описание инструмента (Tool Definition) для Cohere v2. В API v2 инструменты описываются строго в формате JSON Schema<br>
**Step 3. Запуск цикла общения с моделью (Tool Use Loop)**<br>
Теперь отправляем запрос модели. Модель увидит, что вопрос касается свежих данных, и вместо ответа вернет команду вызвать инструмент internet_search<br>

In [22]:
import json
import cohere

# 1. Initialize the modern V2 client
co = cohere.ClientV2(api_key=CO_API_KEY)

# Simulated web search engine
def web_search_provider(query: str):
    print(f"🔎 Searching the web for: '{query}'")
    return [
        {
            "id": "web_doc_1",
            "title": "Cohere API updates 2025-2026",
            "snippet": "Cohere deprecated built-in connectors on September 15, 2025. V2 API uses Tool Use."
        }
    ]


In [23]:
# 2. Описываем инструмент для модели
# Define the search tool using strict JSON Schema format
tools = [
    {
        "type": "function",
        "function": {
            "name": "internet_search",
            "description": "Searches the live internet for up-to-date documents and web articles.",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {
                        "type": "string",
                        "description": "The exact query to search for on the web."
                    }
                },
                "required": ["search_query"]
            }
        }
    }
]


In [24]:
# Initialize chat history
messages = [
    {"role": "user", "content": "Who is more popular: Nsync or Backstreet Boys?"}
]

# --- FIRST CALL: Let the model decide if it needs to search ---
response = co.chat(
    model="command-a-03-2025", 
    messages=messages,
    tools=tools
)

# --- TRACK AND APPEND THE ASSISTANT'S TOOL CALL CORRECTLY ---
if response.message.tool_calls:
    # 1. Append the model's assistant action response exactly as a structured dict
    messages.append({
        "role": "assistant",
        "tool_calls": [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            } for tc in response.message.tool_calls
        ]
    })
    
    # 2. Execute the function for each requested tool call
    for tool_call in response.message.tool_calls:
        if tool_call.function.name == "internet_search":
            args = json.loads(tool_call.function.arguments)
            query_from_llm = args.get("search_query")
            
            # Run the search
            search_results = web_search_provider(query_from_llm)
            
            # 3. Append the execution result back to the thread
            # CRITICAL: 'tool_call_id' must strictly match the id requested by the model
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id, 
                "content": json.dumps(search_results, ensure_ascii=False)
            })
            
    # --- SECOND CALL: Generate the final answer using the retrieved documents ---
    final_response = co.chat(
        model="command-a-03-2025",
        messages=messages
    )
    
    print("\n🤖 Final Answer:")
    print(final_response.message.content[0].text)

else:
    print(response.message.content[0].text)


🔎 Searching the web for: 'nsync popularity'
🔎 Searching the web for: 'backstreet boys popularity'

🤖 Final Answer:
[
    {"tool_call_id": "2", "tool_name": "internet_search", "parameters": {"search_query": "Nsync vs Backstreet Boys popularity"}}
]
